# CNN 실습 코드 모음
실습 코드가 하나의 노트북에 통합되어 있습니다.<br>
원하는 파트의 코드를 찾으실 때는 좌측 목차 탭을 활용하시면 편리합니다!


In [1]:
import torch
from torch import nn

print("PyTorch Version:", torch.__version__)

PyTorch Version: 2.14.1+cpu


# 01. CNN 기본 개념



In [2]:
import torch
from torch import nn

def count_params(model):  # 모델의 학습 가능한 값을 세는 함수
    return sum(p.numel() for p in model.parameters())
   # model.parameters(): 모델 내 가중치/편향 텐서들을 순서대로 반환 → p.numel(): 각 텐서의 원소 개수 계산

x = torch.zeros(1, 1, 5, 5)
# 이미지는 `[N, C, H, W]` 4개 축이 필요 → (N)이미지 1장, (C)채널 1개(흑백), (H)높이 5, (W)너비 5

fc = nn.Linear(25, 9, bias=False)  # 완전연결층 : 입력 25개 → 출력 9개 (가중치 : [9, 25])

conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
# (입력 채널 1, 출력 채널(필터) 1, 필터 크기 3 x 3, 편향 사용 X)

print(fc(x.flatten(start_dim=1)).shape)
print(conv(x).shape)
print(count_params(fc), count_params(conv))

torch.Size([1, 9])
torch.Size([1, 1, 3, 3])
225 9


# 02. 합성곱 계산과 특징맵

In [3]:
import torch
from torch import nn

x = torch.tensor([[1., 2., 0., 1.],
                  [3., 1., 1., 2.],
                  [2., 0., 4., 0.],
                  [1., 3., 2., 1.]])  # 숫자 뒤에 .을 붙여 float로 만듦 (Conv2d는 정수 Tensor를 받으면 타입 오류)

x = x.reshape(1, 1, 4, 4)
# [N, C, H, W] → (N)이미지 1장, (C)채널 1개(흑백), (H)높이 4, (W)너비 4

K = torch.tensor([[ 1.,  1.,  1.],
                  [ 0.,  0.,  0.],
                  [-1., -1., -1.]])  # 직접 정한 3×3 필터

conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
# (입력 채널 1, 출력 채널(필터 개수) 1, 필터 크기 3 x 3, 편향 사용 X) → 필터 값은 처음엔 랜덤

print(conv.weight.shape)

with torch.no_grad():
    conv.weight.copy_(K.reshape(1, 1, 3, 3))  # copy_ : 끝에 _가 붙은 함수는 원본을 직접 바꿈

out = conv(x)
print(out.shape)
print(out[0, 0].detach())  # 앞의 두자리 빼고 뒤의 특징맵만 호출

torch.Size([1, 1, 3, 3])
torch.Size([1, 1, 2, 2])
tensor([[-3., -1.],
        [-1., -2.]])


# 03. Stride와 Padding

In [4]:
import torch
from torch import nn

x = torch.zeros(1, 1, 5, 5)  # [N, C, H, W] → 이미지 1장, 흑백, 5×5 ('0'으로 구성)

for stride, padding in [(1, 0), (2, 0), (1, 1), (2, 1)]:
    conv = nn.Conv2d(1, 1, kernel_size=3, stride=stride, padding=padding, bias=False)
    out = conv(x)  # 입력 x를 합성곱 층에 통과시켜 출력 특징맵 계산
    print(f"stride={stride}, padding={padding} -> {tuple(out.shape)}")

stride=1, padding=0 -> (1, 1, 3, 3)
stride=2, padding=0 -> (1, 1, 2, 2)
stride=1, padding=1 -> (1, 1, 5, 5)
stride=2, padding=1 -> (1, 1, 3, 3)


# 04. Dilation

In [5]:
import torch
from torch import nn

x = torch.tensor([[2., 7., 1., 0., 3.],
                  [5., 9., 4., 8., 6.],
                  [1., 3., 2., 7., 0.],
                  [4., 6., 5., 1., 9.],
                  [0., 8., 3., 2., 4.]]).reshape(1, 1, 5, 5)
K = torch.tensor([[ 1.,  1.,  1.],
                  [ 0.,  0.,  0.],
                  [-1., -1., -1.]])

for d in (1, 2):
    conv = nn.Conv2d(1, 1, kernel_size=3, dilation=d, bias=False)  # dilation 인자만 추가 (기본값 1)
    with torch.no_grad():
        conv.weight.copy_(K.reshape(1, 1, 3, 3))
    out = conv(x)
    print(f"d={d} -> weight {tuple(conv.weight.shape)}, out {tuple(out.shape)}, first {out[0, 0, 0, 0].item()}")
    # weight 모양은 d와 상관없이 그대로 (1, 1, 3, 3) → 가중치 9개 / out은 k_eff 때문에 작아짐

d=1 -> weight (1, 1, 3, 3), out (1, 1, 3, 3), first 4.0
d=2 -> weight (1, 1, 3, 3), out (1, 1, 1, 1), first -1.0


# 05. Channel과 Conv2d

In [6]:
import torch
from torch import nn

x = torch.tensor([[[1., 0., 2.], [3., 1., 0.], [0., 2., 1.]],
                  [[2., 1., 0.], [0., 1., 3.], [1., 0., 2.]],
                  [[0., 2., 1.], [1., 0., 0.], [2., 1., 1.]]]).reshape(1, 3, 3, 3)
# [N, C, H, W] → 이미지 1장, 채널 3개(R, G, B), 3×3
K = torch.tensor([[[ 1., 0., -1.], [ 1., 0., -1.], [ 1., 0., -1.]],
                  [[ 0., 0.,  0.], [ 0., 1.,  0.], [ 0., 0.,  0.]],
                  [[ 1., 1.,  1.], [ 0., 0.,  0.], [-1., -1., -1.]]]).reshape(1, 3, 3, 3)
# [Cout, Cin, kH, kW] → 필터 1개, 채널마다 3×3 가중치

conv = nn.Conv2d(3, 1, kernel_size=3)  # bias=True가 기본값 → 출력 필터마다 bias 1개
with torch.no_grad():
    conv.weight.copy_(K)  # K가 이미 weight와 같은 4축 모양
    conv.bias.fill_(2.)   # bias [1]을 2로 채움

out = conv(x)
print(conv.weight.shape, conv.bias.shape, out.shape, out.item())
# out.item() : 값이 하나뿐인 Tensor를 숫자로 꺼냄

torch.Size([1, 3, 3, 3]) torch.Size([1]) torch.Size([1, 1, 1, 1]) 3.0


In [7]:
layer = nn.Conv2d(3, 4, kernel_size=3, padding=1)

for shape in [(2, 3, 8, 8), (8, 3, 32, 32)]:
    y = layer(torch.zeros(*shape))  # 입력 크기만 바꿔도 같은 layer(같은 가중치)를 사용
    print(shape, '->', tuple(y.shape))

print(layer.weight.shape, layer.bias.shape)
print(sum(p.numel() for p in layer.parameters()))  # weight 108 + bias 4
print((y[0, 2] == layer.bias[2]).all().item())  # 입력이 0이면 출력 채널 2의 모든 칸이 bias[2]

(2, 3, 8, 8) -> (2, 4, 8, 8)
(8, 3, 32, 32) -> (8, 4, 32, 32)
torch.Size([4, 3, 3, 3]) torch.Size([4])
112
True
